# Basic Planet Structure with Magrathea

This notebook demonstrates how to use the Magrathea Python package to model the internal structure of planets.

## Installation

First, ensure Magrathea is installed. In Google Colab or local environments:

In [ ]:
# Uncomment to install in Colab (after PyPI release)
# !pip install magrathea

# For local development, install from repository root:
# pip install -e .

## Import Libraries

In [ ]:
import magrathea as mag
import numpy as np
import matplotlib.pyplot as plt

# Configure matplotlib
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"Magrathea version: {mag.__version__}")

## Create an Earth-like Planet

Let's model a planet similar to Earth with a 33% core and 67% mantle by mass.

In [ ]:
# Create a new planet
earth = mag.Planet()

# Add layers (masses in Earth masses)
earth.add_layer('core', mass=0.33)
earth.add_layer('mantle', mass=0.67)

# Set surface temperature (Kelvin)
earth.surface_temp = 300

# Display planet configuration
print(earth.summary())

## Solve for Internal Structure

Now we'll solve the hydrostatic equilibrium equations to determine the planet's internal structure.

In [ ]:
# Solve (this may take a few seconds)
try:
    results = earth.solve()
    print(results.summary())
except Exception as e:
    print(f"Error: {e}")
    print("\nNote: This requires the Magrathea C++ executable to be built.")
    print("From the repository root, run: make")

## Plot Density Profile

Visualize how density varies with enclosed mass.

In [ ]:
if earth.results is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(earth.results.mass, earth.results.density, 'b-', linewidth=2)
    ax.axvline(x=0.33, color='r', linestyle='--', alpha=0.5, label='Core-Mantle Boundary')
    
    ax.set_xlabel('Enclosed Mass (M⊕)', fontsize=14)
    ax.set_ylabel('Density (g/cm³)', fontsize=14)
    ax.set_title('Density Profile of Earth-like Planet', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Central density: {earth.results.density[0]:.2f} g/cm³")
    print(f"Surface density: {earth.results.density[-1]:.2f} g/cm³")
else:
    print("No results to plot. Solve the planet structure first.")

## Plot Pressure Profile

In [ ]:
if earth.results is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(earth.results.mass, earth.results.pressure, 'g-', linewidth=2)
    ax.axvline(x=0.33, color='r', linestyle='--', alpha=0.5, label='Core-Mantle Boundary')
    
    ax.set_xlabel('Enclosed Mass (M⊕)', fontsize=14)
    ax.set_ylabel('Pressure (GPa)', fontsize=14)
    ax.set_title('Pressure Profile of Earth-like Planet', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Central pressure: {earth.results.pressure[0]:.2f} GPa")
    print(f"Surface pressure: {earth.results.pressure[-1]:.2e} GPa")

## Compare Different Compositions

Let's create and compare planets with different core mass fractions.

In [ ]:
# Create planets with different core mass fractions
core_fractions = [0.1, 0.33, 0.5, 0.7]
planets = []
radii = []

for core_frac in core_fractions:
    p = mag.Planet()
    p.add_layer('core', mass=core_frac)
    p.add_layer('mantle', mass=1.0 - core_frac)
    p.surface_temp = 300
    
    try:
        p.solve()
        planets.append(p)
        radii.append(p.results.radius if p.results.radius else 0)
        print(f"Core fraction {core_frac:.2f}: Radius = {p.results.radius:.4f} R⊕")
    except:
        print(f"Core fraction {core_frac:.2f}: Failed to solve")
        radii.append(0)

# Plot radii vs core fraction
if any(r > 0 for r in radii):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(core_fractions, radii, 'bo-', linewidth=2, markersize=8)
    
    ax.set_xlabel('Core Mass Fraction', fontsize=14)
    ax.set_ylabel('Planet Radius (R⊕)', fontsize=14)
    ax.set_title('Planet Radius vs Core Composition', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Summary

In this notebook, we:

1. Created a planet model using Magrathea's Pythonic API
2. Solved for the internal structure
3. Visualized density and pressure profiles
4. Compared planets with different compositions

Magrathea makes it easy to explore planetary interiors through a simple, REBOUND-like interface!